# ML Workflow with DVC

This notebook is a guided workshop for building a minimal, reproducible machine learning workflow with **Data Version Control (DVC)** using the project structure below.

```text
project/
├── data/
├── src/
│   ├── prepare.py
│   ├── train.py
├── params.yaml
├── dvc.yaml
```

The workflow assumes you are working in Visual Studio Code and running commands from the project root folder.


## 1. Introduction to DVC

**DVC stands for Data Version Control.  
It is an open‑source tool designed for managing machine learning and data science projects, especially when you work with large datasets, models, and experiments that don’t fit well into Git alone.**

In the workplace, DVC is useful when teams need to coordinate code, data, models, and results without placing large binary assets directly inside a Git repository. In many organizations, software engineers, data scientists, and researchers work together across branches, machines, and cloud environments. DVC helps these teams keep a lightweight Git history while storing the actual datasets and trained models in external storage. This makes it easier to reproduce results, compare experiments, onboard new team members, and reduce confusion about which dataset or model version produced a particular result. DVC is especially valuable in research labs, applied ML teams, and MLOps pipelines where traceability and repeatability matter.

**Official DVC portal:** <https://dvc.org/>  
**Official documentation:** <https://doc.dvc.org/>


## 2. What is the role of DVC in the MLOps workflow?

DVC sits at the intersection of **code versioning, data management, experiment tracking, and reproducibility**. In a modern MLOps workflow, it helps connect what was run, on which data, with which parameters, and what outputs were produced.

### What problem does DVC solve?

Git works very well for **code**, but machine learning projects often include assets that Git alone does not handle elegantly:

- Large datasets
- Model checkpoints and trained models
- Intermediate artifacts
- Repeatable experiment pipelines
- Clear traceability between code, data, and results

DVC complements Git by versioning the **metadata** about large files and pipeline outputs while storing the heavy artifacts elsewhere.

### What DVC does

DVC helps you:

#### ✅ Version datasets and models
- Keeps lightweight metadata (`.dvc` files) in Git
- Stores actual data in external storage (local disk, S3, Azure Blob, GCP, SSH, etc.)

#### ✅ Track experiments
- Records parameters, metrics, and outputs
- Allows you to compare experiments (like a Git diff for ML)

#### ✅ Ensure reproducibility
- Defines pipelines so models can be rebuilt from raw data
- Tracks dependencies between data, code, and models

#### ✅ Collaborate effectively
- Team members can reproduce results exactly
- Everyone gets the same data/model versions tied to a Git commit


## 3. Build a Minimal DVC Project

In this section, students build and run a small DVC project for a CNN trained on MNIST.

### Learning goals

By the end of this section, students should be able to:

- initialize a Git + DVC project
- understand the role of each project file
- run a two-stage DVC pipeline
- inspect metrics
- make a controlled experiment change
- explain why DVC reruns some stages and skips others


### 3.1 Setup commands (Bash)

In [ ]:
# Run these in a Bash terminal from the project root

pip install dvc torch torchvision scikit-learn pandas pyyaml
git init
dvc init


**Explanation**

- `pip install ...` installs the Python packages used in this workshop.
- `git init` creates a Git repository so code and DVC metadata can be versioned.
- `dvc init` creates the DVC configuration inside the repository.


### 3.2 Setup commands (PowerShell)

In [ ]:
# Run these in PowerShell from the project root

pip install dvc torch torchvision scikit-learn pandas pyyaml
git init
dvc init


The commands are the same in PowerShell for this basic setup. Later, file paths may look slightly different on Windows.


### 3.3 Project structure

```text
project/
├── data/
├── src/
│   ├── prepare.py
│   ├── train.py
├── params.yaml
├── dvc.yaml
```

### What each part does

- `data/` stores raw and processed data artifacts.
- `src/prepare.py` downloads and prepares the dataset.
- `src/train.py` trains the CNN and writes outputs for the pipeline.
- `params.yaml` stores experiment settings such as epochs and learning rate.
- `dvc.yaml` defines the DVC pipeline stages, their dependencies, and their outputs.


### 3.4 Create `params.yaml`

In [ ]:
epochs: 2
lr: 0.001
batch_size: 64


This file stores experiment parameters outside the training code. That makes it easier to rerun the pipeline after changing one setting and observe the result.


### 3.5 `src/prepare.py`

In [ ]:
import os
import torch
from torchvision import datasets, transforms

OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="data/raw",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="data/raw",
    train=False,
    download=True,
    transform=transform
)

def dataset_to_tensors(dataset):
    images = []
    labels = []

    for img, label in dataset:
        images.append(img)
        labels.append(label)

    images = torch.stack(images)
    labels = torch.tensor(labels)

    return images, labels

train_images, train_labels = dataset_to_tensors(train_dataset)
test_images, test_labels = dataset_to_tensors(test_dataset)

torch.save((train_images, train_labels), os.path.join(OUTPUT_DIR, "train.pt"))
torch.save((test_images, test_labels), os.path.join(OUTPUT_DIR, "test.pt"))

print("Data preparation complete.")
print(f"Saved to: {OUTPUT_DIR}")


### Role of `prepare.py` in the pipeline

This script is the **data preparation stage**. It:

- downloads MNIST
- converts the dataset into tensors
- saves the processed training and test data to disk

For teaching purposes, this script is intentionally simple. It avoids complex preprocessing so students can focus on the workflow and the role of DVC.


### 3.6 `src/train.py`

In [ ]:
import os
import json
import yaml
import torch
import torch.nn as nn
import torch.optim as optim

DATA_DIR = "data/processed"
MODEL_PATH = "model.pt"
METRICS_PATH = "metrics.json"

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

EPOCHS = params["epochs"]
LR = params["lr"]
BATCH_SIZE = params["batch_size"]

train_images, train_labels = torch.load(os.path.join(DATA_DIR, "train.pt"))
test_images, test_labels = torch.load(os.path.join(DATA_DIR, "test.pt"))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 13 * 13, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    for i in range(0, len(train_images), BATCH_SIZE):
        x_batch = train_images[i:i + BATCH_SIZE]
        y_batch = train_labels[i:i + BATCH_SIZE]

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {loss.item():.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)
    total += test_labels.size(0)
    correct += (predicted == test_labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

torch.save(model.state_dict(), MODEL_PATH)

metrics = {"accuracy": round(accuracy, 4)}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print(f"Model saved to {MODEL_PATH}")
print(f"Metrics saved to {METRICS_PATH}")


### Role of `train.py` in the pipeline

This script is the **training stage**. It:

- loads the processed tensors
- reads hyperparameters from `params.yaml`
- trains a tiny CNN
- saves a trained model to `model.pt`
- writes evaluation results to `metrics.json`

In this workshop, the model is intentionally small so that students can run it quickly on a CPU and focus on reproducibility rather than model complexity.


### 3.7 `dvc.yaml`

In [ ]:
stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
    outs:
      - data/processed

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/processed
    params:
      - epochs
      - lr
      - batch_size
    outs:
      - model.pt
    metrics:
      - metrics.json


### How to read `dvc.yaml`

- `stages` defines the pipeline.
- `prepare` is the first stage and runs `python src/prepare.py`.
- `deps` lists dependencies. If one changes, DVC knows the stage may need to rerun.
- `outs` lists outputs produced by a stage.
- `train` depends on both `src/train.py` and the prepared data directory.
- `params` tells DVC which experiment settings should be tracked from `params.yaml`.
- `metrics` tells DVC which result files should be displayed and compared.

This file is the heart of the pipeline: it defines **what runs, in what order, based on which inputs**.


### 3.8 Run the pipeline

In [ ]:
# Bash

python src/prepare.py
python src/train.py


Optional warm-up: run the scripts directly once so students can see what each stage does individually.

In [ ]:
# Bash

dvc repro
dvc metrics show


In [ ]:
# PowerShell

dvc repro
dvc metrics show


### What students should notice

- DVC runs the stages in dependency order.
- The pipeline creates `data/processed`, `model.pt`, and `metrics.json`.
- `dvc metrics show` reads the accuracy directly from the metrics file.


### 3.9 Example metrics file

In [ ]:
{
  "accuracy": 0.6967
}


### 3.10 Make controlled changes to the experiment

The idea is to change **one thing at a time** and then rerun the pipeline.

#### Example A: Increase the number of epochs


In [ ]:
epochs: 5
lr: 0.001
batch_size: 64


In [ ]:
# Bash or PowerShell

dvc repro
dvc metrics show
dvc metrics diff


#### Example B: Change the learning rate


In [ ]:
epochs: 2
lr: 0.0005
batch_size: 64


In [ ]:
# Bash or PowerShell

dvc repro
dvc metrics show
dvc metrics diff


#### Example C: Change the batch size


In [ ]:
epochs: 2
lr: 0.001
batch_size: 128


In [ ]:
# Bash or PowerShell

dvc repro
dvc metrics show
dvc metrics diff


#### Example D: Re-run with no changes

```bash
dvc repro
```

Students should observe that DVC skips unchanged stages. This is a strong demonstration of dependency-aware workflow execution.


### 3.11 Save the project state in Git

In [ ]:
# Bash

git add data/.gitignore .gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py metrics.json
git commit -m "Add minimal DVC MNIST pipeline"


In [ ]:
# PowerShell

git add data\.gitignore .gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py metrics.json
git commit -m "Add minimal DVC MNIST pipeline"


## 4. Talking Points

Use these during the workshop discussion.

### 🧠 Talking Point 1
This pipeline is intentionally minimal. The goal is to teach workflow concepts, not advanced model design.

### 🧠 Talking Point 2
DVC works best when code, parameters, data, and outputs are treated as connected parts of one reproducible system.

### 🧠 Talking Point 3
Changing a parameter is not just “running training again.” It becomes a tracked, comparable experiment.

### 🧠 Talking Point 4
When DVC skips a stage, it is showing students that reproducibility is also about **efficiency**.

### 🧠 Talking Point 5
A weak result in a simple model is still educationally valuable if the workflow is transparent and reproducible.

### 🧠 Talking Point 6
This small project mirrors a larger professional pattern: data preparation, model training, metrics, and versioned pipeline logic.

### 🧠 Talking Point 7
Students often think reproducibility starts after model selection. In practice, it starts the moment raw data enters the workflow.

### 🧠 Talking Point 8
DVC does not replace Git. It extends Git-based collaboration into the data and model parts of machine learning work.


### Additional Talking Points for Section 5

#### 0. Important subtle issue (VERY useful for teaching)
Importing a class from another Python file only works cleanly when that file separates **definitions** from **execution**. If `train.py` runs training code as soon as it is imported, then importing `SimpleCNN` inside prediction code may accidentally trigger training again.

#### 1. Best practice (recommended fix)
A good pattern is to place the training workflow inside a `main()` function and then run it only with:

```python
if __name__ == "__main__":
    main()
```

This allows other files to import `SimpleCNN` safely without causing the training script to execute.

#### 2. Teaching insight (very valuable moment)
This is a useful example of a broader software engineering principle in machine learning: separate **model definitions** from **script execution**. Students often write everything in one file at first, but reusable ML workflows benefit from modular code.

#### 3. Important note about imports in the notebook vs the pipeline
The prediction example in **Section 5.1** can run in the notebook with:

```python
from src.train import SimpleCNN
```

because the notebook is usually executed from the project root, where `src` is visible as a package-like folder.

However, this will not necessarily run as-is when the same code is saved to `src/predict.py` and executed from the pipeline with:

```bash
python src/predict.py
```

For that script-based example, the import may need to be refactored to:

```python
from train import SimpleCNN  # Assuming train.py is in the same directory for this example
```

This distinction is worth highlighting explicitly: **execution context affects how imports work**.


## 5. Expand the Pipeline: Add a Classification Stage

So far, the pipeline prepares data and trains a CNN. A natural next step is to **use the trained model to run classifications** and save prediction outputs.

In this extension, students add a new script called `src/predict.py`. This script loads:

- the trained model from `model.pt`
- the processed test data from `data/processed/test.pt`

It then runs inference and saves a small results file. This demonstrates an important MLOps idea: **training is not the end of the workflow**. In practice, models are trained so they can be used for prediction, evaluation, and downstream decision-making.

### Updated pipeline idea

```text
prepare  →  train  →  predict
```

The new `predict` stage depends on both the trained model and the prediction script. DVC will only rerun this stage when one of its dependencies changes.

This is a good example of how pipelines grow over time: first data preparation, then training, then inference.


### 5.1 Create `src/predict.py`

In [ ]:
import json
import os
import torch

# Import the model class from train.py
# IMPORTANT: Ensure that the SimpleCNN class is defined in src/train.py and is accessible here
# and also when the code is run from a Python script, the current working directory should be the project root
from src.train import SimpleCNN

DATA_PATH = os.path.join("data", "processed", "test.pt")
MODEL_PATH = "model.pt"
OUTPUT_PATH = "predictions.json"

# Load data
test_images, test_labels = torch.load(DATA_PATH)

# Instantiate model
model = SimpleCNN()

# Load trained weights
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

Epoch 1/2, Loss: 1.8043
Epoch 2/2, Loss: 0.7924
Test Accuracy: 0.8249
Model saved to model.pt
Metrics saved to metrics.json


SimpleCNN(
  (conv): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=1352, out_features=10, bias=True)
)

### 5.2 Update `dvc.yaml`

Add a third stage to the pipeline:

```yaml
predict:
  cmd: python src/predict.py
  deps:
    - src/predict.py
    - model.pt
    - data/processed/test.pt
  outs:
    - predictions.json
```

With this addition, DVC understands that prediction depends on the trained model and the processed test data. If the model changes because training is rerun, the `predict` stage will also rerun.


### 5.3 Update `src/predict.py`

Add code to run inferences (see TODO section below)

In [ ]:
import json
import os
import torch

# Import the model definition from train.py
# IMPORTANT: Ensure that the SimpleCNN class is defined in src/train.py and is accessible here
# and also when the code is run from a Python script, the current working directory should be the project root
from src.train import SimpleCNN

DATA_PATH = os.path.join("data", "processed", "test.pt")
MODEL_PATH = "model.pt"
OUTPUT_PATH = "predictions.json"

# Load data
test_images, test_labels = torch.load(DATA_PATH)

# Initialize model and load weights
model = SimpleCNN()
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

# TODO: Add code to run inference and save predictions 
with torch.no_grad():
    outputs = model(test_images)
    predicted = torch.argmax(outputs, dim=1)

# Save a small sample of predictions
sample_results = []
for i in range(10):
    sample_results.append({
        "index": i,
        "true_label": int(test_labels[i].item()),
        "predicted_label": int(predicted[i].item())
    })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sample_results, f, indent=4)

print(f"Saved predictions to {OUTPUT_PATH}")


### 5.3 Run the expanded pipeline

Once `src/predict.py` and the updated `dvc.yaml` are in place, students can rerun the pipeline from the project root.

They should observe that DVC now executes three stages:

- `prepare`
- `train`
- `predict`

and produces a new output file:

- `predictions.json`

That file contains a small sample of predicted labels from the trained model.


In [ ]:
# Bash

dvc repro
type predictions.json


In [ ]:
# PowerShell

dvc repro
Get-Content predictions.json


### 5.4 Try a follow-up experiment

Change a value in `params.yaml`, rerun the pipeline, and inspect whether the predictions change.

For example:

- increase `epochs`
- change `lr`
- change `batch_size`

Then run:

```bash
dvc repro
dvc metrics show
```

Students should notice that once training changes, the prediction stage also reruns because it depends on `model.pt`.

This reinforces a key workflow lesson: **pipeline stages are connected by dependencies, not by manual memory**.


### 5.5 Share artifacts with DVC remote storage and push code to Git

Once the pipeline has produced data, model files, metrics, and predictions, the next practical step is to **share the large artifacts with DVC remote storage** and **share the code and metadata with Git**.

This reflects a common real-world workflow:

- **Git** stores code and lightweight project metadata
- **DVC remote storage** stores large artifacts such as processed data and model outputs

In this exercise, students should understand that `git push` and `dvc push` serve different purposes and are often used together.

#### What should be available before pushing?

Before pushing, make sure you have:

- run the pipeline successfully
- configured a Git remote repository
- configured a DVC remote location
- staged and committed the code and DVC metadata locally

#### Typical sequence

1. Run the pipeline  
2. Stage and commit code plus DVC metadata  
3. Push large artifacts with `dvc push`  
4. Push code and metadata with `git push`

This sequence helps keep code, pipeline definitions, and data artifacts aligned.


#### Configure Google Drive as DVC Remote

This project uses **Google Drive** as the DVC remote storage backend. Large artifacts such as processed datasets, the trained model, and prediction outputs are pushed to a shared Google Drive folder rather than committed to Git.

**What you need before configuring the remote:**

1. A Google account
2. A Google Cloud project with the **Google Drive API** enabled
3. An **OAuth 2.0 Desktop client ID and client secret** from the Google Cloud Console (APIs & Services → Credentials → Create Credentials → OAuth client ID → Desktop app)
4. Your Google account email added as a **test user** in the OAuth consent screen (APIs & Services → OAuth consent screen → Test users)

**Security note:** The `gdrive_client_secret` must always be stored with `--local` so it is written to `.dvc/config.local`, which is gitignored and never committed to the repository. The remote URL and client ID are safe to commit via `.dvc/config`.


In [ ]:
# Bash

# Step 1: Install the DVC Google Drive extension
pip install "dvc[gdrive]"

# Step 2: Add the Google Drive remote using your shared folder ID
# The folder ID comes from the Google Drive URL: drive.google.com/drive/folders/FOLDER_ID
dvc remote add -d myremote gdrive://YOUR_FOLDER_ID

# Step 3: Set your OAuth client ID (safe to commit to Git via .dvc/config)
dvc remote modify myremote gdrive_client_id 'YOUR_CLIENT_ID'

# Step 4: Set your OAuth client secret (stored locally only, never committed)
dvc remote modify --local myremote gdrive_client_secret 'YOUR_CLIENT_SECRET'

# Step 5: Confirm the remote is registered
dvc remote list


In [ ]:
# PowerShell

# Step 1: Install the DVC Google Drive extension
pip install "dvc[gdrive]"

# Step 2: Add the Google Drive remote using your shared folder ID
dvc remote add -d myremote gdrive://YOUR_FOLDER_ID

# Step 3: Set your OAuth client ID (safe to commit to Git via .dvc/config)
dvc remote modify myremote gdrive_client_id 'YOUR_CLIENT_ID'

# Step 4: Set your OAuth client secret (stored locally only, never committed)
dvc remote modify --local myremote gdrive_client_secret 'YOUR_CLIENT_SECRET'

# Step 5: Confirm the remote is registered
dvc remote list


#### First-time authorization and common errors

The first time `dvc push` or `dvc pull` runs against the Google Drive remote, DVC opens a browser window asking you to sign in and grant access to the configured Google Drive folder. After you approve, DVC saves a token locally and subsequent operations do not require re-authorization.

**Common errors and fixes:**

| Error | Cause | Fix |
|---|---|---|
| `No module named 'dvc_gdrive'` | Google Drive extension not installed | `pip install "dvc[gdrive]"` |
| `This app is blocked` | OAuth app is unverified by Google | Use your own OAuth credentials (client ID + secret) instead of DVC's built-in app |
| `Error 403: access_denied` | Your account is not a test user | Go to Google Cloud Console → APIs & Services → OAuth consent screen → Test users → add your email |
| `Expected credentials type 'service_account'` | `gdrive_use_service_account` is set but the JSON file is a Desktop OAuth credential, not a service account key | Remove `gdrive_use_service_account` with `dvc remote modify myremote --unset gdrive_use_service_account` and use `gdrive_client_id` / `gdrive_client_secret` instead |

**What gets stored where:**

- `.dvc/config` — remote URL and client ID (committed to Git, safe to share)
- `.dvc/config.local` — client secret (gitignored, stays on this machine only)
- `.dvc/tmp/gdrive-user-credentials.json` — OAuth token after first login (gitignored)


#### Stage and commit the project before pushing

Students should stage and commit the latest code and DVC metadata before pushing to shared locations.


In [ ]:
# Bash

git add .dvc/config data/.gitignore .gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py src/predict.py metrics.json predictions.json
git commit -m "Add prediction stage, Google Drive remote, and updated DVC pipeline"


In [ ]:
# PowerShell

git add .dvc\config data\.gitignore .gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py src/predict.py metrics.json predictions.json
git commit -m "Add prediction stage, Google Drive remote, and updated DVC pipeline"


#### Push large artifacts to the DVC remote

This uploads the tracked data and model artifacts to the configured DVC remote.


In [ ]:
# Bash or PowerShell

dvc push
dvc status -c


`dvc status -c` compares the local cache with the configured remote and helps students verify whether the remote is synchronized.


#### Push the code and metadata to the Git remote

This shares the repository history, pipeline definition, parameter file, and lightweight DVC metadata with collaborators.


In [ ]:
# Bash

git remote add origin <your-repo-url>
git branch -M main
git push -u origin main


In [ ]:
# PowerShell

git remote add origin <your-repo-url>
git branch -M main
git push -u origin main


If the default branch is named `master` instead of `main`, replace `main` with `master`.

On a fresh repository, students may also need to create or rename the branch before the first push.


In [ ]:
# Bash or PowerShell

git branch -M main
git push -u origin main


#### Suggested verification commands

These commands help confirm that both code and artifacts are in the expected state.


In [ ]:
# Bash or PowerShell

git status
dvc status
dvc status -c
dvc metrics show


#### Why both commands matter

- `git push` shares the **code, notebook, pipeline file, parameter file, and DVC metadata**
- `dvc push` shares the **large tracked artifacts**

Students should leave this section understanding that a collaborator often needs **both**:

1. the Git repository
2. access to the DVC remote

That combination allows another person to clone the code, pull the artifacts, and reproduce the workflow.


#### End-to-end collaboration example

On another machine, a collaborator would typically run:

```bash
git clone <repository>
cd <repository-folder>
pip install dvc "dvc[gdrive]" torch torchvision scikit-learn pandas pyyaml

# Set the OAuth client secret locally (the client ID is already in .dvc/config from Git)
dvc remote modify --local myremote gdrive_client_secret 'YOUR_CLIENT_SECRET'

# Pull large artifacts from Google Drive (triggers browser auth on first run)
dvc pull

# Re-run the pipeline (skips stages whose inputs are unchanged)
dvc repro
dvc metrics show
```

**Key point:** The collaborator only needs to supply their own `gdrive_client_secret` locally. The remote URL and `gdrive_client_id` are already committed in `.dvc/config` and come down with `git clone`. This is one of the clearest demonstrations of reproducible machine learning workflow in practice — code, pipeline definition, and artifact pointers all travel together through Git, while the heavy binary files travel through DVC remote storage.
